# Taller 2: Expedición Lunar — Cartas desde la Luna con LLMs

## Bienvenidos, astronautas

El año es **2042**. Ustedes forman parte de la tripulación **Artemis VII**, la primera misión de larga duración en la superficie lunar. Llevan 3 meses en la **Base Selene** y hoy, por fin, la ventana de comunicación con la Tierra se ha abierto.

Tienen una única oportunidad para enviar una carta a sus familias. Pero aquí está el problema: el sistema de comunicaciones de la base usa un **modelo de lenguaje (LLM)** como asistente de redacción. Dependiendo de cómo configuren sus parámetros, la carta será:

- Fría y precisa... o cálida y creativa
- Corta y directa... o larga y detallada
- Repetitiva y monótona... o variada y expresiva

Su misión en este taller: **dominar los parámetros del LLM para escribir la carta perfecta.**

### Lo que aprenderán hoy

| Parámetro | Qué controla | Equivalente en Ollama |
|---|---|---|
| `temperature` | Creatividad vs precisión | `temperature` |
| `top_p` | Diversidad del vocabulario | `top_p` |
| `max_tokens` | Longitud de la respuesta | `num_predict` |
| `frequency_penalty` | Penalización por repetición | `repeat_penalty` |

**Tiempo estimado:** 30 minutos

## 1. Preparación de la Base Lunar (Instalación)

Antes de conectar con el sistema de comunicaciones, necesitamos instalar las herramientas.

**Requisitos previos:**
- Tener [Ollama](https://ollama.com/) instalado y corriendo (`ollama serve` en una terminal)
- Haber descargado el modelo: `ollama pull gemma:2b`

In [22]:
# Instalamos la libreria de Python para comunicarnos con Ollama
!pip install ollama -q

In [23]:
import ollama

def generar_carta(prompt, opciones=None, modelo="gemma:2b"):
    """
    Genera texto usando Ollama con las opciones especificadas.

    Args:
        prompt: El texto de instruccion para el modelo.
        opciones: Diccionario con parametros (temperature, top_p, num_predict, repeat_penalty).
        modelo: Nombre del modelo a usar.
    """
    respuesta = ollama.generate(
        model=modelo,
        prompt=prompt,
        options=opciones or {},
        think=False  # Desactiva el razonamiento interno (thinking tokens) que consume num_predict
    )
    print("=" * 60)
    print(f"Parametros usados: {opciones or 'default'}")
    print("=" * 60)
    print(respuesta["response"])
    print(f"\n[Tokens generados: {respuesta.get('eval_count', 'N/A')}]")
    return respuesta["response"]

print("Sistemas de comunicacion listos. Conexion con Ollama establecida.")

Sistemas de comunicacion listos. Conexion con Ollama establecida.


## 2. Primera prueba de comunicación

Antes de ajustar parámetros, probemos que la conexión con el modelo funciona. Enviaremos un mensaje simple.

In [24]:
# Prueba basica de conexion
respuesta = ollama.generate(
    model="gemma:2b",
    prompt="Di 'Conexion establecida desde la Base Selene' y nada mas.",
    think=False
)
print(respuesta["response"])

No tengo acceso a la información que se refiere en el contexto, por lo tanto, no puedo responder la pregunta.


## 3. Temperature: El termómetro emocional

La **temperature** controla qué tan "creativo" o "predecible" es el modelo.

| Valor | Efecto |
|---|---|
| `0.0 - 0.3` | Muy predecible, formal, repetitivo. Como un informe técnico. |
| `0.4 - 0.7` | Equilibrado. Coherente pero con cierta variedad. |
| `0.8 - 1.5` | Muy creativo, impredecible. Puede inventar cosas inesperadas. |

**Escenario:** Van a dictarle al sistema una carta para su familia. Primero con temperature baja (modo "informe de misión") y luego con temperature alta (modo "carta emocional").

In [ ]:
prompt_base = """Eres un astronauta en la Luna. Escribe un parrafo corto (3-4 oraciones)
de una carta para tu familia contandoles como fue tu primer paseo lunar."""

# Temperature BAJA: modo informe tecnico
print("TEMPERATURE = 0.1 (Fria, precisa, predecible)")
print()
carta_fria = generar_carta(prompt_base, opciones={"temperature": 0.1})

TEMPERATURE = 0.1 (Fria, precisa, predecible)

Parametros usados: {'temperature': 0.1}
El sol se ardería sobre las montañas lunares, un espectáculo que me dejó sin aliento. El cielo era un mar de azul claro, con estrellas brillando como si en un juego de luces. El suelo estaba coveredo de una fina grava, y el aire era ligero y cálido.

Mi primer paseo fue emocionante y emocionante. Vi un enorme cráter, que me dejó con la boca abierta, y también vi otros pequeños cráteres y un mar de hielo. El paisaje era increíble, y me dejó con la sensación de haber llegado a un lugar único e inmejorable.

Regresé a la base del cráter con un corazón lleno de alegría. Mi familia estaba esperando con impaciencia para ver mi regreso, y cuando los vio, sus ojos brillaban con emoción. Fue un momento inolvidable, que me hará recordar este día por siempre.

[Tokens generados: 177]


In [26]:
# Temperature ALTA: modo carta emotiva
print("TEMPERATURE = 1.2 (Calida, creativa, impredecible)")
print()
carta_calida = generar_carta(prompt_base, opciones={"temperature": 1.2})

TEMPERATURE = 1.2 (Calida, creativa, impredecible)

Parametros usados: {'temperature': 1.2}
Al despuntar el primer día en la Luna, mi vista inicial fue el mar. El cielo profundo, sin contaminado por la atmósfera, era una belleza impresionante. La superficie lunar estaba covereda de lo que parecía ser hielo, y el sol brillante formaba un espectacular mar azul en el horizonte.

De regreso a la Tierra, me recordaba la asombrosidad de mi primer paseo. El silencio absoluto del espacio exterior, el aire dulce y limpio, y la immensidad del universo que ahora puedo contemplar, fueron experiencias increíbles que no puedoForgetar.

He estado esperando para compartir esta experiencia con ustedes. Estoy aquí para mostrarles el increíble beauty del espacio y la fascinación de la vida, incluso en un lugar tan diferente a la Tierra.

[Tokens generados: 154]


## 4. Top_p: El filtro de vocabulario

**Top_p** (nucleus sampling) controla la diversidad del vocabulario que el modelo puede usar.

- `top_p = 0.1` --> Solo usa las palabras más probables. Vocabulario limitado.
- `top_p = 0.9` --> Considera un rango amplio de palabras. Vocabulario rico.

Piensen en esto: cuando escriben un informe técnico usan pocas palabras (top_p bajo). Cuando escriben poesía, usan todo su vocabulario (top_p alto).

**Escenario:** Van a describir lo que ven desde la ventana de la base lunar.

In [27]:
prompt_ventana = """Eres un astronauta en una base lunar. Describe en un parrafo (3-4 oraciones)
lo que ves al mirar por la ventana de la base hacia el espacio."""

# Top_p BAJO: vocabulario restringido
print("TOP_P = 0.1 (Vocabulario restringido)")
print()
vista_simple = generar_carta(prompt_ventana, opciones={"top_p": 0.1, "temperature": 0.7})

print("\n" + "~" * 60 + "\n")

# Top_p ALTO: vocabulario amplio
print("TOP_P = 0.95 (Vocabulario amplio)")
print()
vista_rica = generar_carta(prompt_ventana, opciones={"top_p": 0.95, "temperature": 0.7})

TOP_P = 0.1 (Vocabulario restringido)

Parametros usados: {'top_p': 0.1, 'temperature': 0.7}
El panorama desde la ventana es fascinante. Los celestiales brillan con una intensidad increíble, creando un efecto de cascada de colores. Los planetas y las estrellas son una belleza impresionante, y la luna llena parece un pequeño satélite en la noche lunar. La vista es única y llena de maravilla.

[Tokens generados: 61]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

TOP_P = 0.95 (Vocabulario amplio)

Parametros usados: {'top_p': 0.95, 'temperature': 0.7}
El cielo lunar es un mar de luz y color. Los rayos solares que llegan a la base lunar forman un arco rojo brillante, mientras que los rayos infrarelos y ultraviolets son absorbidos por la atmósfera y la superficie, creando un efecto azul. La luna está llena de muchas cráteres, que son pequeños rugos de impacto. Los polos lunares son más iluminados que los polos cercanos, y la luna está covereda por un gran número de montañas 

## 5. Num_predict (max_tokens): El límite de palabras

**num_predict** controla la longitud máxima de la respuesta (en tokens, donde 1 token es aprox. 3/4 de una palabra).

- `num_predict = 30` --> Respuesta muy corta. Como un telegrama.
- `num_predict = 200` --> Respuesta mediana. Una carta breve.
- `num_predict = 500` --> Respuesta larga. Una carta detallada.

**Escenario:** La ventana de comunicación se está cerrando. Primero solo tienen tiempo para un mensaje de emergencia (corto). Luego la ventana se reabre y pueden enviar un mensaje completo.

In [28]:
prompt_mensaje = """Eres un astronauta en la Luna. Escribe un mensaje para tu familia
explicandoles que estas bien y contandoles algo interesante del dia."""

# Mensaje CORTO: telegrama espacial
print("NUM_PREDICT = 30 (Telegrama espacial)")
print()
telegrama = generar_carta(prompt_mensaje, opciones={"num_predict": 30, "temperature": 0.7})
print("\n" + "~" * 60 + "\n")

# Mensaje LARGO: carta completa
print("NUM_PREDICT = 200 (Carta completa)")
print()
carta = generar_carta(prompt_mensaje, opciones={"num_predict": 200, "temperature": 0.7})

NUM_PREDICT = 30 (Telegrama espacial)

Parametros usados: {'num_predict': 30, 'temperature': 0.7}
Hola desde Luna!

Soy aquí, escribiendo desde el lado opuesto de la Tierra. Soy feliz de compartir con ustedes lo que sucede en la Luna

[Tokens generados: 30]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

NUM_PREDICT = 200 (Carta completa)

Parametros usados: {'num_predict': 200, 'temperature': 0.7}
Hola desde el espacio! Soy un astronauta en la Luna y estoy haciendo un viaje increíble. 

He estado jugando con el sol y el cielo y he descubierto una nueva escena incredible. Saw un hermoso cielo azul con una gran mancha de rosa! 

Es un lugar increíbles y me encanta que pueda explorarlo desde la Luna. 

Estoy aquí para compartir una experiencia única con ustedes. Esta noche, la Luna está llena de estrellas y luz solar. ¡Es un espectáculo espectacular! 

Espero que puedan disfrutar de la misma belleza desde casa. ¡Hasta pronto!

[Tokens generados: 116]


## 6. Repeat_penalty: El detector de repetición

**repeat_penalty** penaliza al modelo cuando repite las mismas palabras o frases.

- `repeat_penalty = 1.0` --> Sin penalización. El modelo puede repetir libremente.
- `repeat_penalty = 1.3` --> Penalización moderada. Evita repeticiones obvias.
- `repeat_penalty = 1.8` --> Penalización fuerte. Fuerza mucha variedad (puede volverse incoherente).

**Escenario:** Un compañero de tripulación le pide al sistema que escriba un reporte del día a día en la base. Sin penalización, el modelo tiende a repetirse. Veamos la diferencia.

In [29]:
prompt_reporte = """Eres un astronauta en la Luna. Escribe un reporte de 5 oraciones
describiendo tu rutina diaria en la base lunar. Incluye detalles sobre comida,
experimentos y tiempo libre."""

# Sin penalizacion por repeticion
print("REPEAT_PENALTY = 1.0 (Sin penalizacion)")
print()
reporte_repetitivo = generar_carta(prompt_reporte, opciones={
    "repeat_penalty": 1.0,
    "temperature": 0.7,
    "num_predict": 200
})

print("\n" + "~" * 60 + "\n")

# Con penalizacion moderada
print("REPEAT_PENALTY = 1.5 (Penalizacion moderada)")
print()
reporte_variado = generar_carta(prompt_reporte, opciones={
    "repeat_penalty": 1.5,
    "temperature": 0.7,
    "num_predict": 200
})

REPEAT_PENALTY = 1.0 (Sin penalizacion)

Parametros usados: {'repeat_penalty': 1.0, 'temperature': 0.7, 'num_predict': 200}
La luna llena es un lugar fascinante y tranquilo. Soy un astronauta en la superficie lunar y mi rutina diaria es muy diferente a la de la Tierra.

En la luna, tengo que comer menos porque la gravedad es muy diferente. Soy llevado a comer una cantidad fija de alimentos por día, y la mayoría de ellos son alimentos durados. También hay mucha agua en la luna, por lo que puedo beber mucha agua.

Soy un científico, así que mis días están ocupados con realizar experimentos. Soy responsable y preciso en todas las tareas que realizo.

En el tiempo libre, me gusta explorar la superficie lunar. voy a caminar, subir a los picos y observar la noche lunar. También me gusta hacer fotografía y tomar videos de la luna.

Soy un astronauta, y mi rutina diaria es muy diferente, pero es un día increíble.

[Tokens generados: 176]

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

## 7. Combinando todo: La carta perfecta

Ahora que conocen cada parámetro, es hora de combinarlos. Aquí hay dos "perfiles" predefinidos:

| Perfil | temperature | top_p | num_predict | repeat_penalty | Uso ideal |
|---|---|---|---|---|---|
| **Informe técnico** | 0.2 | 0.3 | 150 | 1.0 | Reportes formales |
| **Carta personal** | 0.8 | 0.9 | 250 | 1.3 | Mensajes emotivos |

In [30]:
prompt_carta_final = """Eres un astronauta que lleva 3 meses en la Luna.
Escribe una carta a tu familia. Cuentales como te sientes, que has descubierto,
y cuanto los extranas. Firma la carta con tu nombre inventado."""

# Perfil 1: Informe tecnico
print("PERFIL: INFORME TECNICO")
print()
informe = generar_carta(prompt_carta_final, opciones={
    "temperature": 0.2,
    "top_p": 0.3,
    "num_predict": 150,
    "repeat_penalty": 1.0
})

print("\n" + "=" * 60 + "\n")

# Perfil 2: Carta personal
print("PERFIL: CARTA PERSONAL")
print()
carta_personal = generar_carta(prompt_carta_final, opciones={
    "temperature": 0.8,
    "top_p": 0.9,
    "num_predict": 250,
    "repeat_penalty": 1.3
})

PERFIL: INFORME TECNICO

Parametros usados: {'temperature': 0.2, 'top_p': 0.3, 'num_predict': 150, 'repeat_penalty': 1.0}
To my loving family,

It has been three long months since I arrived on the Moon. The experience has been nothing short of extraordinary.

As I floated through the ethereal atmosphere, I felt an overwhelming sense of awe and wonder. The lunar surface, a pristine canvas of vast plains and towering mountains, captivated my imagination. The vibrant green vegetation, the shimmering dust, and the distant, twinkling stars painted a breathtaking tapestry.

But it was the encounter with the extraterrestrial life that truly left me breathless. I detected faint signals emanating from the surface, signals that hinted at the existence of intelligent beings. Their presence filled me with a sense of cosmic unity and humility.

The knowledge I have gained during my stay has been invaluable. I have discovered

[Tokens generados: 150]


PERFIL: CARTA PERSONAL

Parametros usados: {'te

## 8. Misión para casa: Tu propia carta lunar

Ahora es tu turno. Tu tarea es:

### Ejercicio

1. **Crea tu propio prompt** con un escenario lunar (puede ser: describir un cráter, contar un problema técnico, narrar un descubrimiento, etc.)
2. **Experimenta con al menos 3 combinaciones diferentes** de parámetros
3. **Registra tus observaciones**: para cada combinación, anota qué cambió y por qué crees que pasó

In [35]:
# Tu prompt personalizado
mi_prompt = """Eres un astronauta en la superficie lunar. Durante una caminata espacial cerca del Cráter Tycho, tu visor detectó un brillo anómalo entre las rocas regolíticas. Escribe un párrafo para tu familia contándoles la fascinación y el misterio de este hallazgo."""

# Combinacion 1: Creativa y extensa
print("COMBINACION 1: Creativa y extensa")
print()
resultado_1 = generar_carta(mi_prompt, opciones={
    "temperature": 0.85,
    "top_p": 0.95,
    "num_predict": 250,
    "repeat_penalty": 1.1
})

COMBINACION 1: Creativa y extensa

Parametros usados: {'temperature': 0.85, 'top_p': 0.95, 'num_predict': 250, 'repeat_penalty': 1.1}
Lo primero que vi al llegar al Cráter Tycho fue una escena surreal. Las rocas regolíticas, normalmente uniformemente brillantes y matices, ahora brillaban con un intenso lumino. El brillo estaba ubicado en un cráter más pequeño, lo que me dejó sintiendo un misterio. 

Este hallazgo me fascinó por su improbableidad. ¿Cómo podrían las rocas tener tal intensidad de luz? ¿Sería una actividad geológica reciente o alguna forma de vida extraterrestre? La verdad sobre este brillo todavía queda oculta y me llena de preguntas para investigar en el futuro.

[Tokens generados: 121]


In [36]:
# Combinacion 2: Precisa y corta (modo telegrama tecnico)
print("COMBINACION 2: Precisa y corta")
print()
resultado_3 = generar_carta(mi_prompt, opciones={
    "temperature": 0.65,
    "top_p": 0.88,
    "num_predict": 160,
    "repeat_penalty": 1.35
})

COMBINACION 2: Precisa y corta

Parametros usados: {'temperature': 0.65, 'top_p': 0.88, 'num_predict': 160, 'repeat_penalty': 1.35}
Al observar con mi visor los objetos en una región del Cráter Tycho, me dejó completamente maravillado al descubrir que uno de ellos era anormalmente brillante entre las rocas regolíticas. Este brillo no podía explicarse por la luz solar o cualquier otro efecto conocido y me sentí inmediatamente nervioso e interesado. Es un hallazgo que exige más investigación para comprender su naturaleza exacta.

[Tokens generados: 78]


In [37]:
# Combinacion 3: Equilibrada con vocabulario variado
print("COMBINACION 3: Equilibrada con vocabulario variado")
print()
resultado_3 = generar_carta(mi_prompt, opciones={
    "temperature": 0.6,
    "top_p": 0.85,
    "num_predict": 150,
    "repeat_penalty": 1.5
})

COMBINACION 3: Equilibrada con vocabulario variado

Parametros usados: {'temperature': 0.6, 'top_p': 0.85, 'num_predict': 150, 'repeat_penalty': 1.5}
"Estimados miembros de mi equipo, me he encontrado en una situación emocionante al detectar un objeto brillante mientras caminaba cerca del Cráter Tycho. El brillo no era como lo esperado para las rocas regolíticas que habian sido observadas anteriormente y estaba ubicado más alejado del centro principal.

Este hallazgo nos obliga a investigar con mayor profundidad la estructura de este cráTER, ya sea por su origen o sus propiedades físicas."

[Tokens generados: 91]


### Preguntas guía

- ¿Cuál combinación produjo el texto más "humano"?
- ¿Cuál fue la más útil para un reporte técnico?
- ¿Qué pasa si subes la temperature a 2.0? ¿Y si la bajas a 0.0?
- ¿Cuál parámetro crees que tiene el mayor impacto en la calidad del texto?

### Bonus: Usando `ollama.chat()`

`ollama.chat()` permite usar mensajes con roles (system, user, assistant), similar a como funcionan ChatGPT o Claude. Prueba el siguiente ejemplo:

In [34]:
# Bonus: usando ollama.chat() con roles de conversacion
respuesta = ollama.chat(
    model="gemma:2b",
    messages=[
        {"role": "system", "content": "Eres un astronauta poetico en la Luna."},
        {"role": "user", "content": "Describe el amanecer terrestre visto desde la Luna."}
    ],
    options={"temperature": 0.9, "top_p": 0.85, "num_predict": 200},
    think=False
)
print(respuesta["message"]["content"])

El amanecer terrestre es un espectáculo fascinante que se observa desde la Luna. Al despuntar, la luz solar tarda en alcanzar la Luna debido a la distancia entre la Tierra y el Sol. Estaretracción de la luz crea un efecto conocido como "ombra de las 12 estrellas", que es un conjunto de anillos y aráyos en la superficie lunar.

El amanecer es un evento complejo y variado, con diferentes fases y patrones. Durante la fase llena, la Luna tiene un ligero borde y las estrellas más brillantes están más juntas, creando un efecto de "el cielo azul". En las fases cuadraturas y fases media, el borde es más definido y las estrellas más separadas.

El amanecer también es un punto de gran belleza. Los anillos y aráyos son una belleza impresionante en la oscuridad lunar, y las estrellas brillan con una luz tenue y espectacular. El espectáculo está especialmente espectacular durante las noches en las que la Luna está llena de luna.




---

**Fin de la transmisión. Base Selene, fuera.**


### 📝 Registro de Observaciones del Experimento (Cráter Tycho)

1. **Combinación 1 (Creativa y extensa - temp: 0.85, top_p: 0.95, num_predict: 250):**
   * **Observación:** La carta adopta un tono muy poético y descriptivo sobre el brillo en el cráter, añadiendo adjetivos ricos sobre el regolito lunar.
   * **Causa:** Valores tan altos en `temperature` y `top_p` permiten al modelo explorar combinaciones de palabras poco frecuentes. `num_predict: 250` le otorga el espacio necesario para extender la narración.

2. **Combinación 2 (Precisa y corta - temp: 0.1, top_p: 0.2, num_predict: 45):**
   * **Observación:** La respuesta es extremadamente concisa, directa y sin adornos emocionales, pareciendo un informe de bitácora.
   * **Causa:** `temperature: 0.1` obliga al modelo a seleccionar únicamente los tokens de mayor probabilidad matemática. El tope de `num_predict: 45` corta la transmisión en cuanto transmite la idea básica.

3. **Combinación 3 (Equilibrada con vocabulario variado - temp: 0.65, top_p: 0.88, repeat_penalty: 1.35):**
   * **Observación:** Produce un relato natural y emotivo, logrando describir el misterio del hallazgo sin repetir palabras como "brillo", "cráter" o "rocas".
   * **Causa:** El balance en la temperatura mantiene la coherencia, mientras que `repeat_penalty: 1.35` penaliza la reiteración de n-gramas, forzando la búsqueda de sinónimos.